

## Setup

In [ ]:
import os as _os
import subprocess as _subprocess

def is_colab_env() -> bool:
  """
  check if we are in Google Colab Hosted Runtime Environment
  """
  try:
    import google.colab
    return True and _os.path.exists("/content")
  except ImportError:
    return False


def get_git_root():
  try:
    return _subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True ).strip()
  except _subprocess.CalledProcessError:
    return None


def colab_repo_sync(repo_url = None, repo_subdir = "."):
  repo_path = get_git_root()
  if repo_path is None:
    if is_colab_env() and repo_url is not None:
      repo_path = "cloned_repo"
      _subprocess.call(['git', 'clone', f'{repo_url}', f'{repo_path}']) # !git clone {repo_url} {repo_path}
      print("Cloning complete.")

      _os.chdir(repo_path) # change directory to cloned repo worktree
  else:
    _os.chdir(repo_path) # change directory to cloned repo worktree

  _os.chdir(repo_subdir) # change directory to the one where this notebook is located in the cloned repo
      

colab_repo_sync(
  repo_url="https://github.com/somombo/sort-bench.git",
  repo_subdir="lab",
)

if is_colab_env(): # check if we are in Google Colab Hosted Runtime Environment
    from env_setup import setup_lean, setup_rust
    setup_lean("v4.24.0") # Then we install the Lean toolchain ~~snd install the Rust (cargo) toolchain~~

    from google.colab import output
    output.enable_custom_widget_manager()
    print("Enabled colab custom widget manager")


print("$PWD:", _subprocess.check_output(['pwd'], text=True))
# _subprocess.call(['git', 'clone', f'{repo_url}', f'{repo_path}'], text=True)

In [ ]:
from impalab_py import Impa
import json

impa = Impa(
  root_dir="..",
)



Let us run a warmup round of the benchmark to pre-build build all the executables

In [ ]:
_build_result = impa.build(
  components_dir="./components",
)
assert _build_result, "Initial build failed"

In [ ]:

_test_results = impa.run(
  generator={
    "name": "somombo_unifshuffle_multi",
    "seed": 42,
    "args": [json.dumps([
      {
        "cardinality": 19,
        "multiplicity": 13,
        "swaps": 2,
        "descending": True,
        "runs": 7,
      }, 
      {
        "cardinality": 17,
        "multiplicity": 11,
        "swaps": None,
        "descending": False,
        "runs": 5,
      }
    ])],
  }, 
  reps=3, 
  tasks=[
    {'executor': 'lean', 'args': ['Somombo.qs.bentleyMcIlroy.classic']},
    {'executor': 'golang', 'args': ['slices.Sort']},
    {'executor': 'cpp', 'args': ['std::sort']},
  ],
  attributes={
    "experiment_name": "Warmup-Experiment",
  },
)

## Experiment Config

In [ ]:
from exploration import ExplorerFromLabDf

ExplorerFromLabDf(_test_results).get_clean_data().head()

In [ ]:
from sample_factors import sample_spaced, get_factors, get_hcns

In [ ]:
import secrets


tasks = [
  # {'executor': 'lean', 'args': ['Somombo.qs.hoare.classic']},
  # {'executor': 'lean', 'args': ['Somombo.pdqsort']},
  {'executor': 'lean', 'args': ['Somombo.qs.bentleyMcIlroy.classic_adapt44']},

  {'executor': 'deno', 'args': ['TypedArray.sort']},
  {'executor': 'rust', 'args': ['slice::sort_unstable']},
  {'executor': 'zig', 'args': ['std.sort.pdq']},
  {'executor': 'cpp', 'args': ['std::sort']},
  {'executor': 'golang', 'args': ['slices.Sort']},
  {'executor': 'java', 'args': ['Arrays.sort']},
  {'executor': 'csharp', 'args': ['std_sort']},

]

generator = {
  "name": "somombo_unifshuffle_multi",
  "seed":  secrets.randbits(64),
}

attributes = {
  "study": "fast_sort_study"
}




## Experiment 1: Cardinality (Pre-sorted in Ascending Order)

In [ ]:
runs = 20
reps = 5

experiment_name = "Experiment 1: Varying Cardinality"

_cards = sample_spaced(
    5,
    start=1000,
    # stop=1500_000, # original
    stop=200_000, 
    geometric=True,
)
print("Geometric Progression of Cardinalities to test:", _cards)

params_list = [
    {"cardinality": card, "multiplicity": 1, "descending": False, 'swaps': None, "runs": runs} for card in _cards
]

total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')


In [ ]:
results1 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp1 = ExplorerFromLabDf(results1)

In [ ]:
exp1.plot_distributions(
  'cardinality', 
  title=f'Performance Distributions vs. Cardinality (Unique Elements: Multiplicity=1)',
  normalized=True,
  ylog=False,
)

In [ ]:
exp1.plot_trends_interactive(
    'cardinality', 
    f'Median Sort Performance vs. Cardinality (Unique Elements: Multiplicity=1)',
    normalized=True,
    xlog=True,
    ylog=False,
)

## Experiment 2: Multiplicity (Fixed Size Pre-sorted in Ascending Order)

In [ ]:
runs = 5
reps = 5

# fixed_size = 166320 # original 
# fixed_size = 110880
fixed_size = 83160
# fixed_size = 55440
# fixed_size = 50400
# fixed_size = 20160
experiment_name = f"Experiment 2 (Varying Multiplicity, Fixed Size = {fixed_size})"

assert fixed_size in get_hcns(), f"The chosen Fixed Size {fixed_size} is not a Highly Composite Number (HCN)"
print(f"Fixed Size: {fixed_size}", )


_multis = sample_spaced(
    count=17,
    population=get_factors(fixed_size),
    geometric=True,
)

print(f'Approximately Geometric Progression of Multiplicities to test that are factors of Fixed Size = {fixed_size} with len={len(_multis)} sample points: \n    {_multis}')

# Vary multiplicity and calculate corresponding cardinality
params_list = []
for multi in _multis:
    if fixed_size % multi == 0: # Ensure multiplicity is an integer
        card = fixed_size // multi
        params_list.append({
            'cardinality': card,
            'multiplicity': multi,
            'descending': False,
            'swaps': None,
            'runs': runs,
        })

assert  len(params_list) == 17

total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')

In [ ]:
results2 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp2 = ExplorerFromLabDf(results2)

In [ ]:
exp2.plot_distributions(
    'multiplicity',
    title=f'Performance Distribution vs. Multiplicity (Fixed Size = {fixed_size})',
    normalized=False,
    ylog=False,
)

In [ ]:
exp2.plot_trends_interactive(
    'multiplicity',
    f'Median Sort Performance vs. Multiplicity (Fixed Size = {fixed_size})',
    normalized=False,
    xlog=True,
    ylog=False,
)

## Experiment 3: Adaptability (Pre-sorted in Ascending Order)

In [ ]:
runs = 10
reps = 5

# fixed_size = 300000 # original
fixed_size = 120000 
experiment_name = "Experiment 3 (Varying Swaps)"

_swaps = [
  # 0, 
  *sample_spaced(
    count=10,
    start=1,
    stop=fixed_size,
    geometric=True,
  )
]
print(f"swaps ({len(_swaps)}):", _swaps)


params_list = [
  {'swaps': swap, 'cardinality': fixed_size, 'multiplicity': 1, 'descending': False, "runs": runs} for swap in _swaps
]

total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')


In [ ]:
results3 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp3 = ExplorerFromLabDf(results3)

In [ ]:
exp3.plot_distributions(
  'swaps', 
  title=f'Performance Distribution vs. Pre-sortedness (Size = {fixed_size}, All Unique Values)',
  normalized=False,
  ylog=False,
)

In [ ]:
exp3.plot_trends_interactive(
  'swaps',
  f'Median Sort Performance vs. Pre-sortedness (Size = {fixed_size}, All Unique Values)', 
  normalized=False, 
  xlog=True, 
  ylog=False,
)

## Experiment 4: Cardinality (Pre-sorted in Descending Order)

In [ ]:
runs = 10
reps = 5

experiment_name = "Experiment 4 (Varying Cardinality Descending)"

_cards = sample_spaced(
    5,
    start=1000,
    # stop=1500_000, # original
    stop=1000_000,
    geometric=True,
)
print("Geometric Progression of Cardinalities to test:", _cards)

params_list = [
    {"cardinality": card, "multiplicity": 1, "swaps": 2, "descending": True, "runs": runs} for card in _cards
]

total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')

In [ ]:
results4 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp4 = ExplorerFromLabDf(results4)

In [ ]:
exp4.plot_distributions(
  'cardinality', 
  title=f'Performance Distributions vs. Cardinality (Unique Elements: Multiplicity=1)',
  normalized=True,
  ylog=False,
)

In [ ]:
exp4.plot_trends_interactive(
    'cardinality', 
    f'Median Sort Performance vs. Cardinality (Unique Elements: Multiplicity=1)',
    normalized=True,
    xlog=True,
    ylog=True,
)

## Experiment 5: Multiplicity (Fixed Size Pre-sorted in Descending Order)

In [ ]:
runs = 10
reps = 5

# fixed_size = 277200 # original
# fixed_size = 166320
fixed_size = 110880
# fixed_size = 83160
# fixed_size = 55440
# fixed_size = 50400
# fixed_size = 20160
experiment_name = f"Experiment 5 (Varying Multiplicity Descending, Fixed Size = {fixed_size})"

assert fixed_size in get_hcns(), f"The chosen Fixed Size {fixed_size} is not a Highly Composite Number (HCN)"
print(f"Fixed Size: {fixed_size}")


_multis = sample_spaced(
    count=17,
    population=get_factors(fixed_size),
    geometric=True,
    # strategy="furthest"
)

print(f'Approximately Geometric Progression of Multiplicities to test that are factors of Fixed Size = {fixed_size} with len={len(_multis)} sample points: \n    {_multis}')

# Vary multiplicity and calculate corresponding cardinality
params_list = []
for multi in _multis:
    if fixed_size % multi == 0: # Ensure multiplicity is an integer
        card = fixed_size // multi
        params_list.append({
            'cardinality': card,
            'multiplicity': multi,
            'descending': True,
            'swaps': 2,
            'runs': runs,
        })

assert  len(params_list) == 17

total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')

In [ ]:
results5 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp5 = ExplorerFromLabDf(results5)

In [ ]:
exp5.plot_distributions(
    'multiplicity',
    title=f'Performance Distribution vs. Multiplicity (Fixed Size = {fixed_size})',
    normalized=False,
    ylog=False,
)

In [ ]:
exp5.plot_trends_interactive(
    'multiplicity',
    f'Median Sort Performance vs. Multiplicity (Fixed Size = {fixed_size})',
    normalized=False,
    xlog=True,
    ylog=False,
)

## Experiment 6: Adaptability (Pre-sorted in Descending Order)

In [ ]:
runs = 15
reps = 5

# fixed_size = 300000 # original
fixed_size = 80000 
experiment_name = "Experiment 6 (Varying Swaps Descending)"

_swaps = [
  0, 
  *sample_spaced(
    count=9,
    start=1,
    stop=fixed_size,
    geometric=True,
  )
]
print(f"swaps ({len(_swaps)}):", _swaps)


params_list = [
  {'swaps': swaps, 'cardinality': fixed_size, 'multiplicity': 1, "descending": True, "runs": runs} for swaps in _swaps
]

total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')


In [ ]:
results6 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp6 = ExplorerFromLabDf(results6)

In [ ]:
exp6.plot_distributions(
  'swaps', 
  title=f'Performance Distribution vs. Pre-sortedness (Size = {fixed_size}, All Unique Values)',
  normalized=False,
  ylog=False,
)

In [ ]:
exp6.plot_trends_interactive(
  'swaps',
  f'Median Sort Performance vs. Pre-sortedness (Size = {fixed_size}, All Unique Values)', 
  normalized=False, 
  xlog=True, 
  ylog=False,
)

## Export Results of Study

In [ ]:
import json
def to_str(exp)-> str:
  return '\n'.join(json.dumps(item) for item in exp.get_raw_data().to_dict(orient='records'))

result1 = to_str(exp1)
result2 = to_str(exp2)
result3 = to_str(exp3)
result4 = to_str(exp4)
result5 = to_str(exp5)
result6 = to_str(exp6)

In [ ]:
import os
os.makedirs("../data", exist_ok=True)

with open(f'../data/{attributes["study"]}.jsonl', 'w', encoding='utf-8') as f:

  f.write('\n'.join([result1, result2, result3, result4, result5, result6])+ '\n')

In [ ]:
with open(f'../data/{attributes["study"]}.jsonl', 'r', encoding='utf-8') as f:

  print(f.read())
